# Phase 1 Base vs SFT Evaluation (Kaggle)

This notebook evaluates a Qwen 0.5B Supervised Fine-Tuned checkpoint against the same model's base version on GSM8K (grade-school math word problems) and MATH (competition mathematics). For each test problem, both models generate one answer with greedy decoding, the final number is extracted with a deterministic parser, and the answer is scored as exactly correct or not. The output is four accuracy numbers (one per checkpoint × dataset pair) plus parse-success rates, per-example details, and a cost summary. The goal is a single number per checkpoint that says whether Supervised Fine-Tuning measurably improved the base model on mathematical reasoning.

This notebook uses `/kaggle/working/` paths and reads `HF_TOKEN` from Kaggle Secrets.

**Links:**
- [Issue 03 PRD](./.scratch/phase1-base-vs-sft-eval/README.md)
- [Issue 02 CLI](./.scratch/phase1-base-vs-sft-eval/issues/02-eval-exact-cli-and-output.md)

In [ ]:
import torch

!nvidia-smi

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


In [ ]:
import os

# Set up paths for Kaggle
REPO_ROOT = '/kaggle/working/finpost'
RESULTS_ROOT = '/kaggle/working/results/evals'
CHECKPOINT_PATH = '/kaggle/working/results/checkpoints/combined_hf_step_3000'

os.makedirs(RESULTS_ROOT, exist_ok=True)
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'RESULTS_ROOT: {RESULTS_ROOT}')
print(f'CHECKPOINT_PATH: {CHECKPOINT_PATH}')

# Read the Hugging Face token from Kaggle Secrets so gated models can
# download on first use.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print("HF_TOKEN loaded from Kaggle Secrets.")
    else:
        print("HF_TOKEN not set in Kaggle Secrets — fine for ungated models.")
except Exception as exc:
    print(f"Could not read HF_TOKEN from Kaggle Secrets: {exc}")

# Clone or update the repo
import subprocess
import sys

if not os.path.exists(REPO_ROOT):
    subprocess.run(['git', 'clone', 'https://github.com/shannan-liu1/finpost.git', REPO_ROOT], check=True)
else:
    subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=False)

# Install in editable mode
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_ROOT], check=True)
print('Repository installed')

In [ ]:
import os
from pathlib import Path

if os.path.isdir(CHECKPOINT_PATH):
    print(f'Checkpoint directory found: {CHECKPOINT_PATH}')
    print('Contents:')
    for item in os.listdir(CHECKPOINT_PATH)[:10]:
        print(f'  {item}')
else:
    if os.path.exists('/kaggle/input'):
        print('Available datasets in /kaggle/input/:')
        for item in os.listdir('/kaggle/input'):
            print(f'  {item}')
    print(f'Checkpoint path not found: {CHECKPOINT_PATH}')
    print('Attach the checkpoint as a Kaggle Dataset or build it in a prior cell')

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

from finpost.evals.sources import REGISTRY

# Smoke test: verify the eval sources are available and loaded correctly
print('Available eval sources:')
for source_name in REGISTRY.keys():
    print(f'  {source_name}')

To populate a dollar-cost estimate in `cost_summary.json`, append `'--gpu-cost-per-hour', '<rate>'` to `cmd`.

In [ ]:
cmd = [
    sys.executable, '-m', 'finpost.evals.eval_exact',
    '--checkpoints', 'base=Qwen/Qwen2.5-0.5B', f'combined={CHECKPOINT_PATH}',
    '--sources', 'gsm8k', 'math',
    '--n', '500',
    '--seed', '42',
    '--out-dir', RESULTS_ROOT,
    '--batch-size-gsm8k', '8',
    '--batch-size-math', '4',
]

result = subprocess.run(cmd, cwd=REPO_ROOT)
if result.returncode != 0:
    print(f'ERROR: eval_exact CLI exited with code {result.returncode}')
else:
    print('Evaluation complete')

In [ ]:
import json
from pathlib import Path

# Load accuracy_summary.json
accuracy_path = Path(RESULTS_ROOT) / 'accuracy_summary.json'
if accuracy_path.exists():
    with open(accuracy_path) as f:
        accuracy_data = json.load(f)

    # Try to render as a DataFrame if pandas is available
    try:
        import pandas as pd
        df = pd.DataFrame(accuracy_data)
        print('Accuracy Summary:')
        print(df.to_string())
    except ImportError:
        # Fallback: stdlib table
        print('Accuracy Summary (pandas not available):')
        if accuracy_data:
            # Print header
            headers = list(accuracy_data[0].keys())
            print(' | '.join(headers))
            print('-' * 80)
            for row in accuracy_data:
                print(' | '.join(str(row.get(h, '')) for h in headers))
else:
    print(f'accuracy_summary.json not found at {accuracy_path}')


In [ ]:
# Load cost_summary.json and print key metrics
cost_path = Path(RESULTS_ROOT) / 'cost_summary.json'
if cost_path.exists():
    with open(cost_path) as f:
        cost_data = json.load(f)

    print('Cost Summary:')
    print(f'  Start: {cost_data.get("start_time", "N/A")}')
    print(f'  End: {cost_data.get("end_time", "N/A")}')
    print(f'  Elapsed seconds: {cost_data.get("elapsed_sec", 0):.1f}')
    print(f'  GPU type: {cost_data.get("gpu_type", "N/A")}')
    print(f'  Generated tokens: {cost_data.get("total_generated_tokens", 0)}')
    print(f'  Tokens/sec: {cost_data.get("tokens_per_second", 0):.2f}')
    if cost_data.get("estimated_cost_usd") is not None:
        print(f'  Estimated cost: ${cost_data.get("estimated_cost_usd", 0):.4f}')
else:
    print(f'cost_summary.json not found at {cost_path}')
